In [30]:
# Load cluster
import pandas as pd
import numpy as np

BEST_CLUSTER = 0

train_df = pd.read_parquet( "../logs/kmean/train_clustered_gen1.parquet" )

test_df = pd.read_parquet( "../logs/kmean/test_clustered_gen1.parquet" )

print( "(Rows, Columns)" )
print( "Train:", train_df.shape )
print( "Test:", test_df.shape )

(Rows, Columns)
Train: (500486, 116)
Test: (150912, 116)


In [31]:
def cluster_composition( df ):
    cluster_summary = df.groupby( "cluster" ).agg(
        rows=( "ticker", "count" ),
        unique_tickers=( "ticker", "nunique" ),
        unique_sectors=( "sector", "nunique" )
    )

    year_counts = (
        df.assign(year=df["date"].dt.year)
          .groupby(["cluster", "year"])
          .size()
          .unstack( fill_value=0 )
    )

    return cluster_summary.join( year_counts )


print( "TRAIN" )
train_cluster_summary = cluster_composition( train_df )
display( train_cluster_summary.sort_values( "rows", ascending=False ) )

print( "\nTEST" )
test_cluster_summary = cluster_composition( test_df )
display( test_cluster_summary.sort_values( "rows", ascending=False ) )

TRAIN


,rows,unique_tickers,unique_sectors,2016,2017,2018,2019
cluster,,,,,,,
1,331265,3019,12,66894,148506,110894,4971
0,169221,3026,12,7851,426,42541,118403



TEST


,rows,unique_tickers,unique_sectors,2020,2021
cluster,,,,,
1,89877,2987,12,16317,73560
0,61035,2992,12,61034,1


In [32]:
# Look at cluster VS average overall
TARGET_COLS = [
    # Absolute returns
    "future_ret_1d",
    "future_ret_1w",
    "future_ret_1m",
    "future_ret_6m",
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y",

    # Relative returns
    "future_excess_1d",
    "future_excess_1w",
    "future_excess_1m",
    "future_excess_6m",
    "future_excess_1y",
    "future_excess_3y",
    "future_excess_5y"
]

ID_COLS = ["ticker", "date", "sector"]
JUNK_EXCLUDE = {
    "price",
    "sector_size",
    "rows",
    "sector_size_market",
    "rows_market"
}

SEC_EXCLUDE = {
    "sector_is_trending",

    "sec_avg_ret_1w",
    "sec_avg_ret_1m",
    "sec_avg_ret_6m",
    "sec_avg_ret_1y",
    "sec_avg_ret_3y",
    "sec_avg_ret_5y",

    "sec_avg_ret_1w_market",
    "sec_avg_ret_1m_market",
    "sec_avg_ret_6m_market",
    "sec_avg_ret_1y_market",
    "sec_avg_ret_3y_market",
    "sec_avg_ret_5y_market",

    "sec_positive_1y_trend_pct",
    "sec_positive_5y_trend_pct",
    "sec_breadth_positive_1y",
    "sec_breadth_positive_5y",

    "sec_ret_1y_dispersion",
    "sec_ret_5y_dispersion",
    "sec_ret_1y_dispersion_market",
    "sec_ret_5y_dispersion_market",

    "quality_score",
    "sector_Unknown",
}

OTHER_EXCLUDE = {
    "pe"
}

numeric_cols = train_df.select_dtypes( include=[np.number] ).columns.tolist()

numeric_cols.remove( "cluster" )

feature_cols = [
    c for c in numeric_cols
    if c not in ( set( ID_COLS ) | set( TARGET_COLS ) | JUNK_EXCLUDE | SEC_EXCLUDE | {c for c in train_df.columns if c.startswith("future_")} | OTHER_EXCLUDE )
]

overall = train_df[feature_cols].mean()

cluster = train_df.groupby("cluster")[feature_cols].mean()

print( "Comparison of cluster average to overall average, per feature")

for c in cluster.index:
    print("\nCLUSTER", c)

    diff = (
        cluster.loc[c] - overall
    ).sort_values(ascending=False)

    print(diff.head(15))

Comparison of cluster average to overall average, per feature

CLUSTER 0
spy_ret_3y                       0.047686
ret_3y                           0.029129
sector_Energy                    0.012442
sector_Healthcare                0.009807
sector_Basic Materials           0.005751
sector_Communication Services    0.003193
sector_Technology                0.001808
sector_Consumer Cyclical         0.001616
alpha_1y                         0.000091
spy_ret_1d                       0.000034
sec_avg_5y_trend_market         -0.000011
trend_vs_sector_5y              -0.000023
sec_avg_5y_trend                -0.000025
5y_trend                        -0.000048
sec_avg_ret_1d                  -0.000080
dtype: float64

CLUSTER 1
ret_5y                                   0.167846
spy_ret_5y                               0.091853
risk_adjusted_1y                         0.089254
sec_breadth_positive_1y_market           0.081474
risk_adjusted_5y                         0.076367
excess_ret_5y        

In [33]:
print( "Comparison of features, cluster to cluster" )

feature_compare = train_df.groupby("cluster")[
    [
        "ret_1y",
        "ret_5y",
        "excess_ret_1y",
        "excess_ret_5y",
        "monotonic_score",
        "risk_adjusted_1y",
        "5y_drawdown"
    ]
].mean()

feature_compare

Comparison of features, cluster to cluster


,ret_1y,ret_5y,excess_ret_1y,excess_ret_5y,monotonic_score,risk_adjusted_1y,5y_drawdown
cluster,,,,,,,
0,-0.009613,0.447111,-0.065428,-0.221417,0.553018,-0.213210,-0.301191
1,0.202928,0.943530,0.038211,0.003341,0.654037,0.050768,-0.222290


In [34]:
# Compare stock performance
def cluster_performance(df):
    return (
        df.groupby("cluster")
          .agg(
              stocks=("ticker", "nunique"),
              rows=("ticker", "count"),

              median_1y=("future_ret_1y", "median"),
              median_3y=("future_ret_3y", "median"),
              median_5y=("future_ret_5y", "median"),

              median_excess_1y=("future_excess_1y", "median"),
              median_excess_3y=("future_excess_3y", "median"),
              median_excess_5y=("future_excess_5y", "median"),

              positive_5y=("future_ret_5y", lambda x: (x > 0).mean()),
              beats_spy_5y=("future_excess_5y", lambda x: (x > 0).mean())
          )
          .sort_values("median_excess_5y", ascending=False)
    )

print("TRAIN")
cluster_performance(train_df)



TRAIN


,stocks,rows,median_1y,median_3y,median_5y,median_excess_1y,median_excess_3y,median_excess_5y,positive_5y,beats_spy_5y
cluster,,,,,,,,,,
1,3019,331265,0.053293,0.146905,0.273505,-0.066624,-0.318074,-0.570242,0.692464,0.233360
0,3026,169221,0.000435,0.237762,0.266575,-0.132282,-0.342036,-0.677633,0.674751,0.219654


In [35]:
print("TEST")
cluster_performance(test_df)

TEST


,stocks,rows,median_1y,median_3y,median_5y,median_excess_1y,median_excess_3y,median_excess_5y,positive_5y,beats_spy_5y
cluster,,,,,,,,,,
0,2992,61035,0.353523,0.231788,0.551068,0.013052,-0.138042,-0.524069,0.750553,0.310215
1,2987,89877,0.025111,0.044102,0.306072,-0.082028,-0.283455,-0.581179,0.674755,0.268289


In [36]:
# Top 50 stocks in best cluster
def best_50( df ):

    best = df[
        df["cluster"] == BEST_CLUSTER
    ]

    stocks = (
        best.groupby("ticker")
        .agg(
            observations=("ticker","count"),
            avg_future_5y=("future_ret_5y","mean"),
            median_future_5y=("future_ret_5y","median"),
            avg_return_1y=("future_ret_1y","mean"),
            avg_beta=("beta_1y","mean"),
            avg_market_cap=("log_market_cap","mean")
        )
        .sort_values(
            "median_future_5y",
            ascending=False
        )
    )

    return stocks.head(50)

best_50( train_df )

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,
SMCI,55,30.612953,30.200399,0.406229,1.934050,20.863918
CHRD,73,30.355222,29.097154,6.624963,3.921370,19.174998
FLNA,68,15.996838,17.386654,2.274506,2.008125,18.131265
ENPH,41,12.978757,12.826226,3.173543,2.410365,20.926678
LEU,81,14.218559,12.504021,1.188506,1.519856,17.868448
TSLA,58,11.332675,10.933098,2.702839,2.419883,24.917280
AEHR,57,10.977989,9.879699,0.150224,1.613607,17.733380
BLDR,56,9.721137,8.910082,0.568991,2.773467,21.171145
GME,54,9.127810,8.794621,-0.039474,2.343962,20.787336


In [37]:
best_50( test_df )

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,
CLS,20,31.330184,31.940581,0.252069,1.878695,20.542634
ESEA,21,32.363869,29.077254,8.906275,0.976963,16.268028
INOD,19,27.059224,26.632260,3.072732,0.285632,18.035419
BBW,20,23.648634,25.079611,4.935589,1.563872,17.192792
STRL,20,22.019853,22.689618,0.753407,1.614981,19.833182
MSTR,19,23.862962,22.435751,3.675053,1.054611,22.309941
DDS,21,19.282959,20.380631,5.504853,2.043689,19.534756
MOD,19,19.828779,19.615922,1.112459,1.822093,19.647190
UAN,22,20.771314,19.437107,7.386854,2.219106,17.545868


In [38]:
# Cluster stability

cluster_frequency = (
    df.groupby("ticker")["cluster"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
)

cluster_frequency.value_counts()

best_stocks = cluster_frequency[
    cluster_frequency == best_cluster
]

best_stocks.head()

NameError: name 'df' is not defined

In [ ]:
# Winning cluster VS market

comparison = pd.DataFrame({
    "winning_cluster": test_df[test_df.cluster==BEST_CLUSTER]["future_ret_5y"],
    "all_stocks": test_df["future_ret_5y"],
    "SPY": test_df[test_df.ticker == "SPY"]["future_ret_5y"]
})

print( "Comparison of returns" )
comparison.describe()

Comparison of returns


,winning_cluster,all_stocks,SPY
count,65598.000000,150912.000000,51.000000
mean,0.968448,0.797057,0.963987
std,2.135491,1.972052,0.116254
min,-0.999578,-0.999578,0.719934
25%,-0.003783,-0.084476,0.861662
50%,0.533836,0.401812,0.968480
75%,1.296885,1.108734,1.057251
max,53.853503,55.104512,1.177245
